In [5]:
import heapq
import itertools

# --- Goal Check ---
def is_goal(rooms):
    return all(status == "Clean" for status in rooms.values())

# --- Successor Function ---
def get_successors(state, room_order):
    location, rooms = state
    successors = []

    rooms_copy = rooms.copy()

    # Action: Suck
    if rooms_copy[location] == "Dirty":
        rooms_copy[location] = "Clean"
        successors.append(("Suck", (location, rooms_copy)))

    # Move actions
    idx = room_order.index(location)
    if idx > 0:
        successors.append(("Left", (room_order[idx-1], rooms_copy)))
    if idx < len(room_order) - 1:
        successors.append(("Right", (room_order[idx+1], rooms_copy)))

    return successors

# --- Heuristic ---
def heuristic(rooms):
    return sum(1 for status in rooms.values() if status == "Dirty")

# --- A* Search ---
def a_star(initial_state, room_order):
    heap = []
    counter = itertools.count()
    start_loc, start_rooms = initial_state
    heapq.heappush(heap, (heuristic(start_rooms), 0, next(counter), initial_state, []))
    visited = set()

    while heap:
        f, g, _, current_state, path = heapq.heappop(heap)
        location, rooms = current_state

        state_key = (location, tuple(rooms.items()))
        if state_key in visited:
            continue
        visited.add(state_key)

        if is_goal(rooms):
            return path, current_state

        for action, next_state in get_successors(current_state, room_order):
            next_location, next_rooms = next_state
            new_g = g + 1
            new_f = new_g + heuristic(next_rooms)
            heapq.heappush(heap, (new_f, new_g, next(counter), next_state, path + [(location, rooms.copy(), action)]))

    return None, None

# --- Input: Number of Rooms ---
while True:
    try:
        num_rooms = int(input("Enter the number of rooms: "))
        if num_rooms < 1:
            print("Must be at least 1 room.")
            continue
        break
    except ValueError:
        print("Enter a valid number.")

# Generate room names
room_order = [f"R{i+1}" for i in range(num_rooms)]
rooms = {}

# Input room statuses
for room in room_order:
    status = input(f"Enter status of {room} (Clean/Dirty): ").capitalize()
    while status not in ["Clean", "Dirty"]:
        status = input(f"Invalid! Enter status of {room} (Clean/Dirty): ").capitalize()
    rooms[room] = status

# Input starting location
start_location = input(f"Enter starting location ({'/'.join(room_order)}): ").upper()
while start_location not in room_order:
    start_location = input(f"Invalid! Enter starting location ({'/'.join(room_order)}): ").upper()

initial_state = (start_location, rooms.copy())

# --- Run A* ---
solution, final_state = a_star(initial_state, room_order)

# --- Print Steps ---
if solution is None:
    print("\nNo solution found.")
else:
    print("\nInitial State:", rooms)
    for step_num, (loc, room_status, action) in enumerate(solution, 1):
        print(f"\nStep {step_num}")
        print("Agent Location:", loc)
        print("Room Status:", room_status[loc])
        print("Action:", action)
        if action == "Suck":
            room_status[loc] = "Clean"

    print("\nFinal State:", final_state[1])

Enter the number of rooms:  4
Enter status of R1 (Clean/Dirty):  Dirty
Enter status of R2 (Clean/Dirty):  clean
Enter status of R3 (Clean/Dirty):  clean
Enter status of R4 (Clean/Dirty):  Dirty
Enter starting location (R1/R2/R3/R4):  R2



Initial State: {'R1': 'Dirty', 'R2': 'Clean', 'R3': 'Clean', 'R4': 'Dirty'}

Step 1
Agent Location: R2
Room Status: Clean
Action: Left

Step 2
Agent Location: R1
Room Status: Dirty
Action: Right

Step 3
Agent Location: R2
Room Status: Clean
Action: Right

Step 4
Agent Location: R3
Room Status: Clean
Action: Right

Step 5
Agent Location: R4
Room Status: Dirty
Action: Suck

Final State: {'R1': 'Clean', 'R2': 'Clean', 'R3': 'Clean', 'R4': 'Clean'}
